<a href="https://colab.research.google.com/github/liaodid/UniPert-G2CP-baseline/blob/main/hepg2_alignment_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HepG2 药物扰动：转录组—蛋白质组条件审计与小规模处理
版本：2026-10-15。建议 Colab CPU，按顺序运行。先“文件→在云端硬盘中保存副本”，再运行挂载单元。

**研究判断**
- 用户指定 Cell 论文：UniPert-G2CP，DOI 10.1016/j.cell.2026.06.005。作者代码公开列出 LINCS A549/A375/MCF7/PC3/HT29，以及 Cell Painting A549/U2OS；代码中没有 HepG2。Cell Painting 是形态学数据，不是转录组。不能因为论文包含 LINCS 就将其预处理文件视为 HepG2。
- 172 药物蛋白质组：MSV000101671，HepG2，24 h，10 μM，6 个生物学重复/药物。这里优先使用作者发布的处理结果，**不下载原始质谱**。
- 本 notebook 不断言 Cell 论文所有转录条件均为 24 h：需要原文方法与具体元数据逐项核实。
- 为保留 HepG2，另行审计 LINCS GSE92742；其巨型压缩表达矩阵不适合简单按列远程读取，本 notebook **不会下载**该整库。
- 为先跑通较小数据，另设 GSE51952：HepG2、24 h、22 药物、105 张芯片、每条件 3 个生物学重复。它是**替代练习数据，不是 Cell 论文专属数据**，很多剂量与蛋白质组 10 μM 不同。
- 这里是跨研究的条件级对应，不是同一物理细胞或配对个体；不能使用配对检验。没有合格交集也是有效结果。

**本次交付的验证范围**：当前宿主本地执行工具发生路径兼容错误，未读到用户本地 PDF/旧 notebook，也未实际运行 Colab 下载。所有输出由下面真实运行生成，不含预填的匹配数量、模拟表达或虚构结果。

来源：
[Cell](https://doi.org/10.1016/j.cell.2026.06.005)；
[作者复现代码](https://github.com/lynn-1998/UniPert-G2CP_reproduce)；
[Zenodo](https://doi.org/10.5281/zenodo.20355906)；
[蛋白质组方法](https://pmc.ncbi.nlm.nih.gov/articles/PMC13174343/)；
[蛋白质组代码和处理数据](https://github.com/xomicsdatascience/AI-robotics-proteomics)；
[GSE51952](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE51952)；
[LINCS 文件说明](https://clue.io/connectopedia/category/Data)。


In [ ]:
%pip -q install pandas numpy scipy requests pyarrow fsspec matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import io, re, json, gzip, hashlib, time, sys, subprocess
import numpy as np
import pandas as pd
import requests
from urllib.parse import quote
from IPython.display import display

BASE = Path('/content/drive/MyDrive/virtualcell_hepg2_alignment_v1')
RAW, OUT, REVIEW = [BASE / x for x in ('raw', 'results', 'review')]
for p in (RAW, OUT, REVIEW): p.mkdir(parents=True, exist_ok=True)
assert Path('/content/drive/MyDrive').is_dir(), '必须先挂载 Drive，避免写到临时磁盘'
MAX_FILE_MB = 80
MAX_PROTEIN_TOTAL_MB = 200
MAX_DRUGS = 5
RUN_LINCS_METADATA = True
# True 才允许纳入不同剂量进行探索；默认仅处理同为 10 μM 的候选
ALLOW_DOSE_MISMATCH = False
manifest = json.loads((BASE/'downloads.json').read_text()) if (BASE/'downloads.json').exists() else {}
session = requests.Session()
session.headers['User-Agent'] = 'HepG2-thesis-alignment/1.0'

def save_manifest():
    tmp = BASE/'downloads.json.tmp'
    tmp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
    tmp.replace(BASE/'downloads.json')

def download(url, path, max_mb=MAX_FILE_MB):
    path = Path(path)
    cap = int(max_mb*1024**2)
    key = str(path.relative_to(BASE))
    if path.exists() and key in manifest:
        sha = hashlib.sha256(path.read_bytes()).hexdigest()
        if sha == manifest[key]['sha256']:
            return path
        raise ValueError(f'缓存校验失败：{path}；请检查后移走该文件再运行')
    tmp = path.with_name(path.name+'.part')
    n = 0
    try:
        with session.get(url, stream=True, timeout=(30,120)) as r:
            r.raise_for_status()
            if int(r.headers.get('Content-Length',0)) > cap:
                raise ValueError(f'文件超出 {max_mb} MB 上限：{url}')
            with tmp.open('wb') as f:
                for chunk in r.iter_content(1024**2):
                    if not chunk: continue
                    n += len(chunk)
                    if n > cap: raise ValueError('下载体积超限；未自动放宽限制')
                    f.write(chunk)
        tmp.replace(path)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise
    manifest[key] = dict(url=url, bytes=n, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                         retrieved_utc=pd.Timestamp.now(tz='UTC').isoformat())
    save_manifest()
    return path

def save(df, name):
    path = OUT/name
    if path.suffix == '.parquet':
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def get_json(url):
    r = session.get(url, timeout=60); r.raise_for_status(); return r.json()

print('持久化目录：', BASE)
(BASE/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))


Mounted at /content/drive
持久化目录： /content/drive/MyDrive/virtualcell_hepg2_alignment_v1


12707

## 1. 保存论文适用性证据和蛋白质组文件目录
只下载 README、数据加载代码和药物元数据。固定 GitHub 提交版本，避免以后主分支变化造成结果漂移。不会运行仓库中的代码。

In [ ]:
PRO_REPO = 'xomicsdatascience/AI-robotics-proteomics'
UNI_REPO = 'lynn-1998/UniPert-G2CP_reproduce'
def pin_repo(repo):
    pin_file = RAW/(repo.split('/')[-1]+'_commit.json')
    if pin_file.exists(): return json.loads(pin_file.read_text())['sha']
    info = get_json(f'https://api.github.com/repos/{repo}/commits?per_page=1')[0]
    pin_file.write_text(json.dumps({'sha':info['sha'], 'url':info['html_url']}, indent=2))
    return info['sha']
pro_sha, uni_sha = pin_repo(PRO_REPO), pin_repo(UNI_REPO)
download(f'https://raw.githubusercontent.com/{UNI_REPO}/{uni_sha}/README.md', RAW/'unipert_README.md', 1)
download(f'https://raw.githubusercontent.com/{UNI_REPO}/{uni_sha}/G2CP/backbone_model/GEARS/gears/pertdata.py',
         RAW/'unipert_pertdata.py', 1)
text = (RAW/'unipert_pertdata.py').read_text()
lines = [s.strip() for s in text.splitlines() if 'data_name in' in s]
(BASE/'paper_scope_audit.json').write_text(json.dumps({
    'source_commit':uni_sha, 'dataset_declarations':lines,
    'decision':'Published loader does not establish a HepG2 dataset; do not pair other cell lines with HepG2.',
    'paper_all_conditions_24h':'not_verified',
    'proteomics':{'accession':'MSV000101671','cell':'HepG2','time_h':24,'dose_uM':10},
    'alternative_small_transcriptome':'GSE51952',
}, ensure_ascii=False, indent=2))
print('\n'.join(lines))

tree = get_json(f'https://api.github.com/repos/{PRO_REPO}/git/trees/{pro_sha}?recursive=1')
assert not tree.get('truncated'), '目录被截断，需先完整获取目录'
files = pd.DataFrame([x for x in tree['tree'] if x['type']=='blob'])
save(files, 'proteomics_repository_inventory.csv')
meta_path = 'data/drug_meta.parquet'
assert meta_path in set(files.path), '作者目录变化：请查看 inventory，不猜测文件名'
meta = pd.read_parquet(download(
    f'https://raw.githubusercontent.com/{PRO_REPO}/{pro_sha}/{meta_path}',
    RAW/'drug_meta.parquet'))
assert 'drug' in meta.columns
display(meta.head())
print('药物元数据唯一名称数：', meta.drug.nunique())


if data_name in ['LINCS-A549', 'LINCS-A375', 'LINCS-MCF7', 'LINCS-PC3', 'LINCS-HT29', 'LINCS-pool']:
if data_name in ['LINCS-pool']:
elif data_name in ["CPJUMP1-A549", "CPJUMP1-U2OS"]:


AssertionError: 作者目录变化：请查看 inventory，不猜测文件名

In [ ]:
# 先运行原 notebook 的环境、Drive 挂载和函数定义单元格
PRO_REPO = 'jymbcrc/drug-portal'
pro_sha = pin_repo(PRO_REPO)

tree = get_json(
    f'https://api.github.com/repos/{PRO_REPO}/git/trees/{pro_sha}?recursive=1'
)
assert not tree.get('truncated'), '仓库目录不完整'
files = pd.DataFrame(
    x for x in tree['tree'] if x['type'] == 'blob'
)
save(files, 'proteomics_repository_inventory.csv')

meta_path = 'data/drug_meta.parquet'
assert meta_path in set(files['path'])

meta = pd.read_parquet(download(
    f'https://raw.githubusercontent.com/{PRO_REPO}/{pro_sha}/{meta_path}',
    RAW / 'drug_meta.parquet'
))
assert 'drug' in meta.columns

print('药物名称数量：', meta['drug'].nunique())
display(meta.head())

药物名称数量： 170


,drug,target,pathway,research_area,n_sig_proteins_total,n_sig_go_bp_total,n_sig_hallmark_total,protein_fdr,protein_log2fc,go_fdr,hallmark_fdr,ai_summary_md
0,6-Mercaptopurine,Autophagy; Endogenous Metabolite; Nucleoside A...,Autophagy; Cell Cycle/DNA Damage; Metabolic En...,Cancer,586,194,23,0.05,0.5,0.05,0.05,# Drug ID Card: 6-Mercaptopurine\n\n## 1. Basi...
1,ATP (dimagnesium),Endogenous Metabolite,Metabolic Enzyme/Protease,Inflammation/Immunology,160,11,2,0.05,0.5,0.05,0.05,# Drug ID Card: ATP (dimagnesium)\n\n## 1. Bas...
2,Abemaciclib (methanesulfonate),CDK,Cell Cycle/DNA Damage,Cancer,2173,184,13,0.05,0.5,0.05,0.05,# Drug ID Card: Abemaciclib (methanesulfonate)...
3,Acetophenazine (dimaleate),Dopamine Receptor,GPCR/G Protein; Neuronal Signaling,Neurological Disease,396,457,19,0.05,0.5,0.05,0.05,# Drug ID Card: Acetophenazine (dimaleate)\n\n...
4,Adiphenine (hydrochloride),nAChR,Membrane Transporter/Ion Channel; Neuronal Sig...,Neurological Disease,93,3,4,0.05,0.5,0.05,0.05,# Drug ID Card: Adiphenine (hydrochloride)\n\n...


## 2. 下载小型转录组处理矩阵并解析样本条件
GSE51952 的整张小型 Series Matrix 用于读取统一归一化后的值；不下载 CEL/FASTQ。它已包含 105 个样本，下载上限 80 MB。先保留全部样本元数据用于审计，最终仅输出候选条件和所需对照。

标题解析失败时写出原始标题并停止，避免把未知条件自动补成 HepG2/24 h。

In [ ]:
GEO_BASE = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE51nnn/GSE51952'
listing = session.get(GEO_BASE+'/matrix/', timeout=60)
listing.raise_for_status()
matrix_names = sorted(set(re.findall(r'href="([^"/]*series_matrix\.txt\.gz)"', listing.text)))
assert len(matrix_names)==1, f'需要检查 matrix 目录：{matrix_names}'
matrix_file = download(GEO_BASE+'/matrix/'+matrix_names[0], RAW/matrix_names[0])
sample_rows, matrix_lines, table = [], [], False
with gzip.open(matrix_file, 'rt') as f:
    for line in f:
        if line.startswith('!Sample_'):
            sample_rows.append(next(__import__('csv').reader([line.rstrip()], delimiter='\t')))
        elif line.startswith('!series_matrix_table_begin'): table=True
        elif line.startswith('!series_matrix_table_end'): table=False
        elif table: matrix_lines.append(line)
X = pd.read_csv(io.StringIO(''.join(matrix_lines)), sep='\t', index_col=0)
sids = next(row[1:] for row in sample_rows if row[0]=='!Sample_geo_accession')
sample = pd.DataFrame(index=sids)
seen = {}
for row in sample_rows:
    key = row[0].removeprefix('!Sample_')
    seen[key] = seen.get(key,0)+1
    sample[key if seen[key]==1 else f'{key}_{seen[key]}'] = row[1:]
sample.index.name='sample_id'
save(sample.reset_index(), 'GSE51952_sample_metadata_original.csv')
assert set(X.columns)==set(sample.index)
X = X.apply(pd.to_numeric, errors='raise')
pattern = r'^HepG2_(?P<dose>[\d.]+)_(?P<unit>[^_]+)_(?P<drug>.+)_(?P<time>\d+)h_rep(?P<rep>\d+)$'
parsed = sample.title.str.extract(pattern)
if parsed.isna().any(axis=1).any():
    display(sample.loc[parsed.isna().any(axis=1), ['title']])
    raise ValueError('存在无法解析的样本标题，请根据原始 metadata 修正正则')
sample = sample.join(parsed.add_prefix('parsed_'))
sample['cell_line']='HEPG2'
sample['time_h']=pd.to_numeric(sample.parsed_time)
sample['dose_value']=pd.to_numeric(sample.parsed_dose)
sample['dose_unit']=sample.parsed_unit.str.replace('μ','u').str.replace('µ','u')
sample['dose_uM']=np.where(sample.dose_unit.eq('uM'), sample.dose_value, np.nan)
sample['drug_raw']=sample.parsed_drug
# DMSO/PBS 标题末尾编号保留，不把不同对照组任意合并
sample['is_control']=sample.drug_raw.str.match(r'^(DMSO|PBS|D-mannitol)(_|$)', case=False)
assert sample.time_h.eq(24).all() and sample.cell_line.eq('HEPG2').all()
save(sample.reset_index(), 'transcriptome_conditions.csv')
print('矩阵 shape:',X.shape)
display(sample[['title','dose_uM','time_h','is_control']].head())


矩阵 shape: (18988, 105)


,title,dose_uM,time_h,is_control
sample_id,,,,
GSM1255658,HepG2_10000_µM_Acetominophen_24h_rep1,10000.0,24,False
GSM1255659,HepG2_10000_µM_Acetominophen_24h_rep2,10000.0,24,False
GSM1255660,HepG2_10000_µM_Acetominophen_24h_rep3,10000.0,24,False
GSM1255661,HepG2_50_µM_Adefovir_24h_rep1,50.0,24,False
GSM1255662,HepG2_50_µM_Adefovir_24h_rep2,50.0,24,False


## 3. 药物候选匹配与人工身份审计
精确名称及有限同义词用于生成候选，不能证明盐型、立体异构体一致。不使用模糊匹配、不自动删掉 hydrochloride 等盐型。review/identity_review.csv 中按原目录核实后填写 identity_confirmed=True 和证据链接；否则只输出候选。

对照分组必须从样本说明/原文确认，不能把所有 DMSO 样本混合。

In [ ]:
ALIASES = {
    'acetominophen':'acetaminophen',  # 原始 GEO 标题拼写
    'paracetamol':'acetaminophen',
    'cyclosporina':'cyclosporine',
    'cyclosporin a':'cyclosporine',
    'cyclosporine a':'cyclosporine',
}
def drug_key(s):
    s = str(s).replace('_',' ').strip().casefold()
    s = ALIASES.get(s, s)
    return re.sub(r'\s+', ' ', s)

meta['drug_key']=meta.drug.map(drug_key)
sample['drug_key']=sample.drug_raw.map(drug_key)
conditions = sample.loc[~sample.is_control,
    ['drug_raw','drug_key','cell_line','time_h','dose_uM','dose_unit']].drop_duplicates()
pm = meta[['drug','drug_key']].drop_duplicates().rename(columns={'drug':'protein_drug'})
candidates = conditions.merge(pm, on='drug_key', how='inner')
candidates['protein_time_h']=24
candidates['protein_dose_uM']=10.
candidates['same_dose']=np.isclose(candidates.dose_uM,10,atol=1e-6,rtol=0)
candidates['pair_id']=[f'pair_{i:03d}' for i in range(len(candidates))]
save(candidates, 'condition_candidates.csv')
save(conditions.loc[~conditions.drug_key.isin(pm.drug_key)], 'unmatched_transcript_drugs.csv')
review_file = REVIEW/'identity_review.csv'
if not review_file.exists():
    rv=candidates[['pair_id','drug_raw','protein_drug']].copy()
    rv['identity_confirmed']=False
    rv['identity_evidence']=''
    rv.to_csv(review_file,index=False,encoding='utf-8-sig')
review=pd.read_csv(review_file).fillna('')
if not candidates.empty:
    assert not review.pair_id.duplicated().any()
    review['verified'] = review.identity_confirmed.astype(str).str.lower().eq('true') & review.identity_evidence.astype(str).str.strip().ne('')
    candidates=candidates.merge(review[['pair_id','verified']],on='pair_id',how='left')
    candidates['verified']=candidates.verified.fillna(False)
else:
    candidates['verified']=pd.Series(dtype=bool)
strict=candidates.loc[candidates.same_dose & candidates.verified].copy()
exploratory=candidates.loc[~candidates.same_dose & candidates.verified].copy()
save(strict,'strict_condition_matches.csv')
save(exploratory,'dose_mismatched_exploratory_matches.csv')
# 预处理候选最多五药。结果明确保留候选身份，不冒充已确认配对
eligible=candidates if ALLOW_DOSE_MISMATCH else candidates.loc[candidates.same_dose]
selected=eligible.sort_values(['protein_drug','dose_uM']).head(MAX_DRUGS).copy()
save(selected,'selected_conditions.csv')
display(candidates)
print('候选数:',len(candidates),'身份已审核且同剂量:',len(strict),'本次处理条件:',len(selected))
if selected.empty:
    print('没有满足当前条件的候选。保留空结果，继续执行 LINCS 元数据审计；不自动跨剂量或跨细胞系。')


,drug_raw,drug_key,cell_line,time_h,dose_uM,dose_unit,protein_drug,protein_time_h,protein_dose_uM,same_dose,pair_id,verified
0,Tolbutamide,tolbutamide,HEPG2,24,2000.0,uM,Tolbutamide,24,10.0,False,pair_000,False


候选数: 1 身份已审核且同剂量: 0 本次处理条件: 0
没有满足当前条件的候选。保留空结果，继续执行 LINCS 元数据审计；不自动跨剂量或跨细胞系。


## 4. LINCS HepG2 / 24 h / 10 μM 元数据候选
这是为下一步严格对齐寻找更合适的数据。只下载小型元数据，不下载约 20 GB 的压缩 Level 5 全库。输出可提交给后续表达数据子集服务的 sig_id 清单。
L1000 Level 5 是 moderated Z-score，不是 RNA-seq 原始计数或 log2FC，重复已聚合，后续不能把签名当独立生物学重复。

若最终选用 LINCS，需另行获取这些签名的表达子集；本节的候选表本身不代表已取得表达数据。

In [ ]:
lincs_matches = pd.DataFrame()
gene_info = pd.DataFrame()
if RUN_LINCS_METADATA:
    lbase='https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/'
    def lincs_meta(kind):
        fn=f'GSE92742_Broad_LINCS_{kind}.txt.gz'
        return pd.read_csv(download(lbase+fn,RAW/fn),sep='\t',low_memory=False)
    sig=lincs_meta('sig_info')
    pert=lincs_meta('pert_info')
    gene_info=lincs_meta('gene_info')
    for c in ('cell_id','pert_type','pert_itime','pert_idose','sig_id','pert_id'):
        assert c in sig, f'LINCS 缺列 {c}'
    subset=sig.loc[sig.cell_id.astype(str).str.upper().eq('HEPG2') & sig.pert_type.eq('trt_cp')].copy()
    subset['time_h']=pd.to_numeric(subset.pert_itime.astype(str).str.extract(r'^([\d.]+)\s*h$',expand=False),errors='coerce')
    subset['dose_uM']=pd.to_numeric(subset.pert_idose.astype(str).str.replace('µ','u').str.replace('μ','u').str.extract(r'^([\d.]+)\s*uM$',expand=False),errors='coerce')
    subset=subset.loc[subset.time_h.eq(24) & np.isclose(subset.dose_uM,10,atol=1e-6,rtol=0)].copy()
    names=pert[['pert_id','pert_iname']].drop_duplicates()
    assert not names.pert_id.duplicated().any(), '同一 pert_id 多名，先检查'
    subset=subset.drop(columns=['pert_iname'],errors='ignore').merge(names,on='pert_id',how='left',validate='many_to_one')
    subset['drug_key']=subset.pert_iname.map(drug_key)
    lincs_matches=subset.merge(pm,on='drug_key')
    lincs_matches['status']='name_candidate_requires_structure_identity_review'
    save(lincs_matches,'LINCS_HEPG2_24h_10uM_candidate_signatures.csv')
    save(lincs_matches[['sig_id','pert_id','pert_iname','protein_drug']].drop_duplicates(),
         'LINCS_expression_subset_request.csv')
    print('LINCS 候选药物数:', lincs_matches.protein_drug.nunique(), '签名数:',len(lincs_matches))
    display(lincs_matches.head())


LINCS 候选药物数: 0 签名数: 0


,sig_id,pert_id,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id,time_h,dose_uM,pert_iname,drug_key,protein_drug,status


## 5. 小型芯片预处理和对照审计
Series Matrix 应按 GEO 的 data_processing 确认尺度。下面给出分位数及每样本缺失率，不通过数值范围自动猜 log2。请检查原始样本元数据的 data_processing；确认后在配置中填 RNA_SCALE='log2' 或 'linear'，并填写依据。
平台是 Brainarray Entrez CDF，只有明确的数字 ID / 数字_at 才提取为 Entrez；不把任意探针名称当成基因。


In [ ]:
RNA_SCALE = None  # 核对 data_processing 后填写 'log2' 或 'linear'
RNA_SCALE_EVIDENCE = ''
display(X.quantile([0,.01,.5,.99,1]))
qc=pd.DataFrame({'sample_id':X.columns,'missing_fraction':X.isna().mean().values,
                 'median_value':X.median().values})
save(qc,'transcript_sample_qc.csv')
processing_cols=[c for c in sample if c.startswith('data_processing')]
if processing_cols: display(sample[processing_cols].drop_duplicates())
display(sample.loc[sample.is_control,['title','drug_raw']])
control_file=REVIEW/'controls_review.json'
if not control_file.exists():
    control_file.write_text(json.dumps({
        '_instructions':'按 pair_id 填写已核实的 control_ids 和 evidence；不要将不同批次 DMSO 一起使用。',
        'pairs':{}
    },ensure_ascii=False,indent=2))
# 示例结构（不要直接照抄虚构样本）：
# {"pairs":{"pair_000":{"control_ids":["真实GSM编号"],"evidence":"原文/样本页及分组依据"}}}
transcript_diff=pd.DataFrame()
if RNA_SCALE in ('linear','log2') and RNA_SCALE_EVIDENCE.strip() and not selected.empty:
    Y=X.copy()
    if RNA_SCALE=='linear':
        Y=np.log2(Y.where(Y>0))  # 非正数记缺失，不人为加 1
    entrez=Y.index.to_series().astype(str).str.extract(r'^(\d+)(?:_at)?$',expand=False)
    unmapped=pd.DataFrame({'feature':Y.index,'entrez_id':entrez.values})
    save(unmapped.loc[unmapped.entrez_id.isna()],'unmapped_transcript_features.csv')
    Y=Y.loc[entrez.notna()].copy()
    Y.index=entrez.dropna().values
    Y=Y.groupby(level=0).median()
    needed=set(sample.loc[sample.drug_key.isin(selected.drug_key)].index)
    control_review=json.loads(control_file.read_text())['pairs']
    for config in control_review.values(): needed.update(config.get('control_ids',[]))
    Y.loc[:,Y.columns.isin(needed)].to_parquet(OUT/'transcript_selected_log2.parquet')
    from scipy.stats import ttest_ind
    from scipy.stats import false_discovery_control
    rows=[]
    for row in selected.itertuples():
        cfg=control_review.get(row.pair_id,{})
        controls=cfg.get('control_ids',[])
        treatments=sample.index[(sample.drug_key==row.drug_key)&np.isclose(sample.dose_uM,row.dose_uM)].tolist()
        if not controls or not cfg.get('evidence','').strip():
            print(row.pair_id,': 尚未核实对照，跳过差异计算'); continue
        assert set(controls)<=set(sample.index)
        assert sample.loc[controls,'is_control'].all()
        assert sample.loc[controls,'time_h'].eq(24).all()
        assert not set(treatments)&set(controls)
        # PBS/甘露醇不能自动当作药物溶剂对照；由证据明确说明才可选择
        assert len(treatments)>=3 and len(controls)>=3, '每组至少 3 个生物学重复'
        a,b=Y[treatments],Y[controls]
        valid=(a.notna().sum(axis=1)>=3)&(b.notna().sum(axis=1)>=3)
        a,b=a.loc[valid],b.loc[valid]
        p=ttest_ind(a,b,axis=1,equal_var=False,nan_policy='omit').pvalue
        q=np.full(len(p),np.nan); ok=np.isfinite(p)
        q[ok]=false_discovery_control(p[ok],method='bh')
        d=pd.DataFrame({'entrez_id':a.index,'rna_log2FC':a.mean(axis=1)-b.mean(axis=1),
                        'rna_p':p,'rna_fdr':q,'n_rna_treatment':len(treatments),'n_rna_control':len(controls)})
        d['pair_id']=row.pair_id
        rows.append(d)
    if rows: transcript_diff=pd.concat(rows,ignore_index=True)
    save(transcript_diff,'transcript_differential_exploratory.csv')
else:
    print('已完成原始矩阵下载与 QC；RNA 尺度/依据未确认或无候选，不计算 log2FC。')


,GSM1255658,GSM1255659,GSM1255660,GSM1255661,GSM1255662,GSM1255663,GSM1255664,GSM1255665,GSM1255666,GSM1255667,...,GSM1255753,GSM1255754,GSM1255755,GSM1255756,GSM1255757,GSM1255758,GSM1255759,GSM1255760,GSM1255761,GSM1255762
0.00,2.950793,3.016553,2.896107,2.953111,2.980159,2.829034,2.863652,2.967355,2.909537,2.941423,...,2.904405,2.933006,2.863224,2.831528,2.925861,2.877397,2.940285,2.985810,2.751876,2.716326
0.01,3.363962,3.418351,3.350101,3.379321,3.359760,3.375240,3.361790,3.364551,3.363857,3.370289,...,3.376062,3.351848,3.356301,3.387549,3.358658,3.376869,3.389894,3.378328,3.371796,3.395662
0.50,6.877858,6.878269,6.881209,6.828812,6.828484,6.857434,6.860769,6.894761,6.848842,6.857682,...,6.861111,6.822014,6.811674,6.861510,6.861986,6.880471,6.860983,6.907686,6.839740,6.836094
0.99,12.805645,12.820799,12.843269,12.775639,12.777159,12.740355,12.888054,12.903148,12.900176,12.809670,...,12.762736,12.862323,12.822055,12.793189,12.859424,12.750186,12.869105,12.861877,12.813295,12.759470
1.00,14.872366,14.839370,14.850605,14.866706,14.825804,14.889726,14.861320,14.887558,14.836004,14.806984,...,15.037135,15.045817,15.045817,15.008541,14.906285,15.034544,14.947052,14.986898,15.045817,14.988202


,data_processing
sample_id,
GSM1255658,Obtained data sets were re-annotated to the MB...


,title,drug_raw
sample_id,,
GSM1255676,HepG2_2000_µM_D-mannitol_24h_rep1,D-mannitol
GSM1255677,HepG2_2000_µM_D-mannitol_24h_rep2,D-mannitol
GSM1255678,HepG2_2000_µM_D-mannitol_24h_rep3,D-mannitol
GSM1255679,HepG2_0.5_%_DMSO_1_24h_rep1,DMSO_1
GSM1255680,HepG2_0.5_%_DMSO_2_24h_rep1,DMSO_2
GSM1255681,HepG2_0.5_%_DMSO_1_24h_rep2,DMSO_1
GSM1255682,HepG2_0.5_%_DMSO_2_24h_rep2,DMSO_2
GSM1255683,HepG2_0.5_%_DMSO_1_24h_rep3,DMSO_1
GSM1255684,HepG2_0.5_%_DMSO_2_24h_rep3,DMSO_2


已完成原始矩阵下载与 QC；RNA 尺度/依据未确认或无候选，不计算 log2FC。


## 6. 只获取候选药物的蛋白质组处理结果
使用作者 df_ttest_all 的 Parquet 分块。只读取药物列确定相关 row group，再读取相关数据；网络实际传输可能超过最终子集大小（按 row group 存储），不承诺逐药物零额外开销。
如服务器不支持 Range，允许在总预算内下载小型处理分块，不回退到原始质谱/全仓库。保留作者统计，不能据此宣称完成了独立批次校正或差异重分析。


In [ ]:
protein_subset=pd.DataFrame()
if not selected.empty:
    import fsspec
    import pyarrow.parquet as pq
    parts=files.loc[files.path.str.match(r'^data/df_ttest_all(?:_part[^/]*)?\.parquet$')].sort_values('path')
    assert len(parts)>0, '未找到作者差异结果分块，请检查 repository_inventory'
    assert parts['size'].sum()<=MAX_PROTEIN_TOTAL_MB*1024**2, '处理分块总大小超预算，需进一步按药物定位文件'
    wanted=set(selected.protein_drug)
    collected=[]
    audit=[]
    for item in parts.itertuples():
        url=f'https://raw.githubusercontent.com/{PRO_REPO}/{pro_sha}/{quote(item.path)}'
        # 仅小分块（总量受上面限制），下载到 Drive 校验并缓存，读取后只保存目标行
        # 这样无需依赖 GitHub Range，且断线重跑不重下。
        local=download(url, RAW/Path(item.path).name)
        pf=pq.ParquetFile(local)
        assert 'drug' in pf.schema_arrow.names
        for i in range(pf.num_row_groups):
            names=pf.read_row_group(i,columns=['drug']).to_pandas().drug.astype(str)
            if names.isin(wanted).any():
                frame=pf.read_row_group(i).to_pandas()
                collected.append(frame.loc[frame.drug.astype(str).isin(wanted)])
        audit.append({'path':item.path,'bytes':item.size})
    if collected: protein_subset=pd.concat(collected,ignore_index=True)
    save(pd.DataFrame(audit),'proteomics_files_read.csv')
    save(protein_subset,'proteomics_selected_author_statistics.parquet')
    assert set(protein_subset.drug)<=wanted
    print('选定药物处理结果行数：',len(protein_subset))
    display(protein_subset.head())
else:
    print('无选定条件，未下载蛋白表达统计分块。')


无选定条件，未下载蛋白表达统计分块。


## 7. 基因映射与条件级连接
优先利用蛋白统计中的 gene_symbol；否则查询单一 UniProt accession 的 primary gene，保留映射证据。多蛋白组、多基因、重复符号等歧义不展开制造伪重复。
RNA 的 Entrez ID 由 LINCS gene_info 注释为 gene symbol，仅接受一对一关系。后续正式论文应固定 HGNC/NCBI 注释版本并审查别名。


In [ ]:
aligned=pd.DataFrame()
mapping=pd.DataFrame()
if not protein_subset.empty and not transcript_diff.empty:
    p=protein_subset.copy()
    assert {'drug','protein','log2FC'}<=set(p.columns)
    if 'gene_symbol' not in p.columns:
        accessions=p.protein.astype(str).drop_duplicates()
        single=accessions[accessions.str.match(r'^[A-Z0-9]+(?:-\d+)?$')]
        maps=[]
        for start in range(0,len(single),50):
            chunk=single.iloc[start:start+50].tolist()
            query=' OR '.join('accession:'+a for a in chunk)
            url='https://rest.uniprot.org/uniprotkb/stream?'+requests.compat.urlencode(
                {'query':query,'format':'tsv','fields':'accession,gene_primary'})
            cache=RAW/f'uniprot_{hashlib.sha256(query.encode()).hexdigest()[:16]}.tsv'
            f=download(url,cache,10)
            tab=pd.read_csv(f,sep='\t').rename(columns={'Entry':'protein','Gene Names (primary)':'gene_symbol'})
            maps.append(tab)
            time.sleep(.15)
        mapping=pd.concat(maps,ignore_index=True) if maps else pd.DataFrame(columns=['protein','gene_symbol'])
        mapping=mapping.drop_duplicates()
        # 不将 isoform 无依据折叠成 canonical；未命中的保留为未映射
        ambiguous=mapping.protein.duplicated(keep=False)
        mapping=mapping.loc[~ambiguous]
        p=p.merge(mapping,on='protein',how='left',validate='many_to_one')
    save(p[['protein','gene_symbol']].drop_duplicates(),'protein_gene_mapping.csv')
    valid=p.gene_symbol.notna() & p.gene_symbol.astype(str).str.match(r'^[A-Za-z0-9.-]+$')
    save(p.loc[~valid], 'protein_unmapped_or_ambiguous.csv')
    p=p.loc[valid].copy()
    p['gene_symbol']=p.gene_symbol.str.upper()
    p=p.loc[~p.duplicated(['drug','gene_symbol'],keep=False)].copy()
    assert not gene_info.empty, '请启用 LINCS 元数据以取得 Entrez→symbol 注释'
    assert {'pr_gene_id','pr_gene_symbol'}<=set(gene_info.columns)
    gm=gene_info[['pr_gene_id','pr_gene_symbol']].dropna().astype(str).drop_duplicates()
    gm=gm.rename(columns={'pr_gene_id':'entrez_id','pr_gene_symbol':'gene_symbol'})
    gm['gene_symbol']=gm.gene_symbol.str.upper()
    gm=gm.loc[~gm.duplicated('entrez_id',keep=False)&~gm.duplicated('gene_symbol',keep=False)]
    r=transcript_diff.copy(); r.entrez_id=r.entrez_id.astype(str)
    r=r.merge(gm,on='entrez_id',validate='many_to_one')
    r=r.merge(selected[['pair_id','protein_drug','dose_uM','same_dose','verified']],on='pair_id',validate='many_to_one')
    aligned=r.merge(p.rename(columns={'drug':'protein_drug','log2FC':'protein_log2FC'}),
                    on=['protein_drug','gene_symbol'],validate='many_to_one')
    aligned['cell_line']='HEPG2'; aligned['time_h']=24; aligned['protein_dose_uM']=10
    aligned['alignment_status']=np.where(aligned.same_dose & aligned.verified,
         'condition_matched_cross_study_author_protein_stats',
         'exploratory_or_identity_unverified')
    save(aligned,'aligned_gene_responses_exploratory.parquet')
    strict_aligned=aligned.loc[aligned.same_dose & aligned.verified]
    save(strict_aligned,'aligned_gene_responses_condition_matched.parquet')
    print('基因响应连接行数：',len(aligned),'严格条件行数：',len(strict_aligned))
else:
    print('尚无同时具备已审核 RNA 对照/尺度及蛋白结果的条件，不生成虚假连接。')


尚无同时具备已审核 RNA 对照/尺度及蛋白结果的条件，不生成虚假连接。


## 8. 输出报告和初步比较
只在实际获得共同基因后计算每药物 Spearman 相关。不同剂量/身份未确认的结果单独标记。RNA—蛋白响应方向一致性并不证明因果关系；蛋白来源为作者预处理结果，批次混杂和缺失值填补可能影响结论。正式分析需获取所选药物原始定量列及同批次溶剂对照，用可辨识的统计设计重算。


In [ ]:
summary=[]
if not aligned.empty:
    from scipy.stats import spearmanr
    import matplotlib.pyplot as plt
    for pid,d in aligned.groupby('pair_id'):
        v=d[['rna_log2FC','protein_log2FC']].replace([np.inf,-np.inf],np.nan).dropna()
        rho=spearmanr(v.rna_log2FC,v.protein_log2FC).statistic if len(v)>=20 else np.nan
        nonzero=(v.rna_log2FC!=0)&(v.protein_log2FC!=0)
        concordance=(np.sign(v.loc[nonzero,'rna_log2FC'])==np.sign(v.loc[nonzero,'protein_log2FC'])).mean()
        summary.append({'pair_id':pid,'n_genes':len(v),'spearman_rho':rho,
                        'sign_concordance_nonzero':concordance,
                        'status':d.alignment_status.iloc[0]})
        fig,ax=plt.subplots(figsize=(5,4))
        ax.scatter(v.rna_log2FC,v.protein_log2FC,s=5,alpha=.3)
        ax.set(xlabel='RNA log2FC',ylabel='Protein log2FC (author processed)',
               title=f'{pid}; n={len(v)}; rho={rho:.2f}')
        fig.tight_layout(); fig.savefig(OUT/f'{pid}_response.png',dpi=160); plt.close(fig)
save(pd.DataFrame(summary),'per_condition_exploratory_summary.csv')
report={
 'run_time_utc':pd.Timestamp.now(tz='UTC').isoformat(),
 'paper_preprocessed_HepG2_compatibility':'not_established_do_not_substitute_other_cells',
 'small_transcriptome_accession':'GSE51952',
 'protein_accession':'MSV000101671',
 'candidate_conditions':len(candidates),'strict_conditions':len(strict),
 'selected_conditions':len(selected),'LINCS_candidate_signatures':len(lincs_matches),
 'RNA_differential_rows':len(transcript_diff),'protein_selected_rows':len(protein_subset),
 'aligned_gene_rows':len(aligned),
 'strict_expression_alignment_complete':bool(not aligned.empty and (aligned.same_dose & aligned.verified).any()),
 'limitations':[
  'Original local PDF and notebook were not read in authoring environment.',
  'GSE51952 is an alternative small microarray dataset, not the Cell paper processed subset.',
  'LINCS section downloads metadata only; expression subset retrieval is a subsequent task.',
  'Drug identities, RNA scale and solvent-control groupings require evidence in review files.',
  'Cross-study condition alignment does not mean paired biological samples.',
  'Protein statistics are author processed; batch-aware independent reanalysis remains necessary.'
 ]}
(BASE/'run_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
save_manifest()
display(pd.Series(report,dtype=object))
print('所有下载缓存、审核表、输出均在：',BASE)
print('再次运行可复用缓存。编辑审核表后，从相关单元重新运行；不要把空交集改写为成功。')


,0
run_time_utc,2026-10-15T08:04:24.143926+00:00
paper_preprocessed_HepG2_compatibility,not_established_do_not_substitute_other_cells
small_transcriptome_accession,GSE51952
protein_accession,MSV000101671
candidate_conditions,1
strict_conditions,0
selected_conditions,0
LINCS_candidate_signatures,0
RNA_differential_rows,0
protein_selected_rows,0


所有下载缓存、审核表、输出均在： /content/drive/MyDrive/virtualcell_hepg2_alignment_v1
再次运行可复用缓存。编辑审核表后，从相关单元重新运行；不要把空交集改写为成功。
